# 00b - Grid check: do the data need to be reprojected to a regular lat-lon grid?

For every dataset of the configuration (notebook 00), this notebook reads the grid **as stored in the files** and says:
- the **type of grid**: regular lat-lon, Gaussian in latitude (typical of the EC-Earth3 `gr` grid), or something else
  (e.g. a rotated-pole RCM grid, with 2-D latitude / longitude);
- resolution, extent (global or regional), orientation (descending latitude, longitudes 0-360);
- which datasets share **exactly** the same grid.

Orientation differences (descending latitude, 0-360 longitudes) are **not** a reprojection: `ba_utils.read_raw` only
reorders the coordinates. A reprojection (interpolation / conservative regridding) is only needed when the grid itself differs.

Nothing is modified. Only **one file per dataset** is opened (all the years of a dataset share the same grid),
one per piece when a variable is split in pieces (e.g. northern + southern hemisphere), and only its first time step is read.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import glob
import numpy as np, pandas as pd, xarray as xr
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
cfg = bu.load_config(CONFIG_PATH)
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

## 1. Type of grid of each dataset

In [ ]:
def gaussian_lats(n):
    """Latitudes of a global Gaussian grid with n latitudes (roots of the Legendre polynomial), ascending."""
    x, _ = np.polynomial.legendre.leggauss(n)
    return np.rad2deg(np.arcsin(x))


def describe(da):
    lat, lon = da.lat.values, da.lon.values
    dla, dlo = np.diff(lat), np.diff(lon)
    reg_lat = np.ptp(dla) < 1e-3 * abs(dla.mean())
    reg_lon = np.ptp(dlo) < 1e-3 * abs(dlo.mean())
    gauss = (not reg_lat) and lat.size > 10 and np.abs(lat - gaussian_lats(lat.size)).max() < 0.01
    if reg_lat and reg_lon:
        kind = f"regular lat-lon {abs(dla.mean()):.4g} x {abs(dlo.mean()):.4g} deg"
    elif gauss and reg_lon:
        kind = f"Gaussian N{lat.size // 2} (lat) x regular {abs(dlo.mean()):.4g} deg (lon)"
    else:
        kind = "irregular / other"
    is_global = (lat.min() < -85 and lat.max() > 85 and (lon.max() - lon.min() + abs(dlo.mean())) > 359)
    return {"grid": kind, "n_lat x n_lon": f"{lat.size} x {lon.size}",
            "lat step min / max": f"{dla.min():.4f} / {dla.max():.4f}",
            "lon step min / max": f"{dlo.min():.4f} / {dlo.max():.4f}",
            "lat range": f"{lat.min():.2f} .. {lat.max():.2f}", "lon range": f"{lon.min():.2f} .. {lon.max():.2f}",
            "extent": "global" if is_global else "regional",
            "stored lat descending": bool(da.attrs.get("orig_lat_desc")),
            "stored lon 0-360": bool(da.attrs.get("orig_lon_0360"))}


def read_one(name, var):
    """First time step of ONE file of the dataset (one file per pattern if the variable is split, e.g. NH + SH)."""
    pattern = cfg["datasets"][name]["patterns"][var]
    files = [sorted(glob.glob(p))[0] for p in (pattern if isinstance(pattern, list) else [pattern]) if glob.glob(p)]
    ds = xr.open_mfdataset(files, combine="by_coords") if len(files) > 1 else xr.open_dataset(files[0])
    vname = next(a for a in cfg["aliases"].get(var, [var]) if a in ds.data_vars)
    da = ds[vname]
    tdim = next(d for d in da.dims if d in ("time", "valid_time"))
    da = da.isel({tdim: 0})
    la, lo = bu._first(da, ("lat", "latitude", "y")), bu._first(da, ("lon", "longitude", "x"))
    if da[la].ndim != 1 or da[lo].ndim != 1:
        raise ValueError(f"2-D latitude / longitude ({da[la].dims})")
    desc, l360 = bool(da[la][0] > da[la][-1]), bool(da[lo].max() > 180)
    da = bu.norm_coords(da)
    da.attrs.update(orig_lat_desc=desc, orig_lon_0360=l360)
    return da


rows, grids = [], {}
for name in cfg["datasets"]:
    for var in VARS:
        if not bu.dataset_available(cfg, name, var):
            rows.append({"dataset": name, "var": var, "grid": "no file yet (path to fill in / data not ready)"})
            continue
        try:
            da = read_one(name, var)
        except ValueError as exc:    # 2-D lat/lon, e.g. a rotated-pole RCM grid
            rows.append({"dataset": name, "var": var, "grid": f"not a 1-D lat/lon grid (rotated / curvilinear?): {exc}"})
            continue
        except Exception as exc:     # the file itself could not be opened
            rows.append({"dataset": name, "var": var, "grid": f"FILE COULD NOT BE OPENED: {type(exc).__name__}: {exc}"})
            print(f"{name}/{var}: {type(exc).__name__}: {exc}")
            continue
        grids[(name, var)] = da
        rows.append({"dataset": name, "var": var, **describe(da)})

grid_table = pd.DataFrame(rows)
grid_table.to_csv(bu.out_dir(cfg, "exploration") / "grid_check.csv", index=False)
with pd.option_context("display.max_columns", None, "display.width", 250, "display.max_colwidth", 70):
    display(grid_table)

## 2. Which datasets share exactly the same grid?

In [ ]:
names = list(grids)
groups = []
for k in names:
    for g in groups:
        if bu.same_grid(grids[g[0]], grids[k]):
            g.append(k); break
    else:
        groups.append([k])
for i, g in enumerate(groups, 1):
    print(f"Grid {i}: {describe(grids[g[0]])['grid']}  ({grids[g[0]].sizes['lat']} x {grids[g[0]].sizes['lon']})")
    for name, var in g:
        print(f"    {name} / {var}")

## 3. How to read the result

- **ERA5**: if the table says *regular lat-lon 0.25 x 0.25*, no reprojection is needed. A descending latitude or
  0-360 longitudes are only an orientation, handled when the data are read.
- **EC-Earth3** (historical and hosing runs, CMIP6 `gr` grid): a *Gaussian* grid has a longitude step that is constant
  but a latitude step that varies slightly (it is not a regular lat-lon grid in the strict sense).
  - plan B (analogue downscaling on the native GCM): the method takes the native grid as input, so no reprojection is
    needed unless the method requires it;
  - plan A: notebook 02 already regrids everything conservatively to a regular 1 deg grid.
  The historical run and the hosing runs must be on the **same** grid (section 2), otherwise F1 cannot be applied
  unchanged to the hosing runs.
- **RCM**: to be checked as soon as the paths are filled in. If the grid "cannot be read as 1-D lat/lon", it is most
  probably a rotated-pole grid (EURO-CORDEX style, `rlat` / `rlon`): it then has to be reprojected onto a regular
  lat-lon grid before the RCM bias adjustment, because the target (GCM ERA5-like) is on the ERA5 0.25 deg grid.